# Lab 4: Foreign Language Translation

## English-to-Romanian Machine Translation Using a Pre-trained Transformer

**Course:** Deep Learning and Natural Language Processing  
**Term:** Fall 2026 – First Bi-term

### Objective

This lab demonstrates machine translation from English to Romanian using a pre-trained sequence-to-sequence Transformer model from Hugging Face.

The implementation uses:

- PyTorch
- NumPy
- Hugging Face Transformers
- Hugging Face Datasets
- A pre-trained English-to-Romanian translation model
- SacreBLEU for translation quality evaluation

The lab includes an explanation of machine translation, implementation of the translation pipeline, evaluation using the BLEU metric, and analysis of the translation results.

In [1]:
%pip install -q transformers datasets evaluate sacrebleu sentencepiece

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device selected:", device)

PyTorch version: 2.10.0+cpu
NumPy version: 1.26.4
CUDA available: False
Device selected: cpu


## Part A: Machine Translation

Machine translation is the process of automatically converting text from one language into another while preserving the meaning of the original sentence. Modern machine translation systems commonly use neural networks, especially Transformer-based encoder-decoder architectures.

The translation process begins with tokenization. The input sentence is broken into smaller units called tokens, which may represent complete words, subwords, or punctuation. Each token is converted into a numerical identifier so that the neural network can process it.

In a sequence-to-sequence Transformer model, the encoder processes the input sentence and creates contextual representations of the words. These representations capture both the meaning of individual words and their relationships with other words in the sentence. The decoder then uses this encoded information to generate the translated sentence one token at a time.

Attention mechanisms are an important part of this process. They allow the model to focus on the most relevant parts of the input sentence while generating each output token. For example, when translating an English sentence into Romanian, the model can determine which English words are most important for producing each Romanian word.

In this lab, a pre-trained English-to-Romanian model from Hugging Face will be used. Because the model has already been trained on large amounts of bilingual text, it can perform translation without training a new neural network from the beginning. The tokenizer converts English sentences into model inputs, and the model generates the corresponding Romanian tokens. These output tokens are then decoded back into readable text.

The quality of the generated translations will be evaluated using the BLEU metric. BLEU compares the machine-generated translation with one or more human reference translations by measuring the overlap of words and short word sequences. A higher BLEU score generally indicates that the machine translation is more similar to the reference translation, although BLEU does not capture every aspect of translation quality.

In [2]:
model_name = "Helsinki-NLP/opus-mt-en-ro"

print("Model selected:", model_name)
print("Device:", device)

Model selected: Helsinki-NLP/opus-mt-en-ro
Device: cpu


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)

print("Tokenizer loaded successfully.")
print("Model loaded successfully.")

config.json: 0.00B [00:00, ?B/s]

c:\Users\dubey\Documents\UC_GitLab\.venv_global\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dubey\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-ro. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/789k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/817k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Tokenizer loaded successfully.
Model loaded successfully.


In [4]:
test_sentence = "This is a machine translation experiment."

inputs = tokenizer(
    test_sentence,
    return_tensors="pt",
    padding=True,
    truncation=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    translated_tokens = model.generate(**inputs)

translated_text = tokenizer.decode(
    translated_tokens[0],
    skip_special_tokens=True
)

print("English:", test_sentence)
print("Romanian:", translated_text)

English: This is a machine translation experiment.
Romanian: Acesta este un experiment de traducere mașină.


In [6]:
dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-ro"
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

c:\Users\dubey\Documents\UC_GitLab\.venv_global\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dubey\.cache\huggingface\hub\datasets--Helsinki-NLP--opus-100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


en-ro/test-00000-of-00001.parquet:   0%|          | 0.00/153k [00:00<?, ?B/s]

en-ro/train-00000-of-00001.parquet:   0%|          | 0.00/66.0M [00:00<?, ?B/s]

en-ro/validation-00000-of-00001.parquet:   0%|          | 0.00/153k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [7]:
print("First test example:")
print(dataset["test"][0])

print("\nTranslation pair:")
print("English :", dataset["test"][0]["translation"]["en"])
print("Romanian:", dataset["test"][0]["translation"]["ro"])

First test example:
{'translation': {'en': 'Managed care thing?', 'ro': 'Cahill ţi-a vorbit vreodată... despre lucru cu controlarea îngrijiri?'}}

Translation pair:
English : Managed care thing?
Romanian: Cahill ţi-a vorbit vreodată... despre lucru cu controlarea îngrijiri?


In [8]:
sample_size = 100

test_sample = dataset["test"].select(range(sample_size))

print("Evaluation sample size:", len(test_sample))

Evaluation sample size: 100


In [9]:
def translate(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_length=128
        )

    translated_text = tokenizer.decode(
        output_tokens[0],
        skip_special_tokens=True
    )

    return translated_text

In [10]:
sample_text = "Artificial intelligence is changing the world."

sample_translation = translate(sample_text)

print("English :", sample_text)
print("Romanian:", sample_translation)

English : Artificial intelligence is changing the world.
Romanian: Inteligenţa artificială schimbă lumea.


In [11]:
predictions = []
references = []

for example in test_sample:
    english_text = example["translation"]["en"]
    reference_text = example["translation"]["ro"]

    predicted_text = translate(english_text)

    predictions.append(predicted_text)
    references.append([reference_text])

print("Number of predictions:", len(predictions))
print("Number of references :", len(references))

Number of predictions: 100
Number of references : 100


In [12]:
for i in range(5):
    print(f"Example {i + 1}")
    print("English   :", test_sample[i]["translation"]["en"])
    print("Reference :", references[i][0])
    print("Prediction:", predictions[i])
    print("-" * 80)

Example 1
English   : Managed care thing?
Reference : Cahill ţi-a vorbit vreodată... despre lucru cu controlarea îngrijiri?
Prediction: Lucru de îngrijire gestionat?
--------------------------------------------------------------------------------
Example 2
English   : He -- he just don't want him horning in on things is all.
Reference : El - el pur și simplu nu-l vreau Hørning la pe lucruri este tot.
Prediction: El - el pur și simplu nu vrea să-l coarne în pe lucruri este tot.
--------------------------------------------------------------------------------
Example 3
English   : She wants to help Olivia.
Reference : Vrea s-o ajute pe Olivia.
Prediction: Vrea s-o ajute pe Olivia.
--------------------------------------------------------------------------------
Example 4
English   : You could just not read it.
Reference : Ai putea să nu citeşti deloc.
Prediction: N-ai putut s-o citeşti.
--------------------------------------------------------------------------------
Example 5
English   : W

In [13]:
bleu = evaluate.load("sacrebleu")

print("SacreBLEU metric loaded successfully.")

SacreBLEU metric loaded successfully.


In [14]:
bleu_result = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU Score:", round(bleu_result["score"], 2))

BLEU Score: 32.75


## Part C: Analysis

### Explanation of the Commands Used in Part B

The implementation begins by importing the required libraries. PyTorch is used to run the neural translation model, NumPy supports numerical operations, the Transformers library provides the tokenizer and pre-trained sequence-to-sequence model, the Datasets library loads the bilingual dataset, and the Evaluate library provides access to the SacreBLEU metric.

The tokenizer is loaded using `AutoTokenizer.from_pretrained(model_name)`. Its role is to convert English text into numerical token IDs that the model can process. The pre-trained translation model is loaded using `AutoModelForSeq2SeqLM.from_pretrained(model_name)`. In this lab, the model is `Helsinki-NLP/opus-mt-en-ro`, which translates English text into Romanian.

The bilingual dataset is loaded with `load_dataset()`. The OPUS-100 English-Romanian dataset contains aligned sentence pairs, with one English sentence and its corresponding Romanian reference translation. A subset of 100 test examples is used for evaluation because the model is running on a CPU and does not require retraining.

The `translate()` function performs the main translation process. First, the English sentence is tokenized and converted to PyTorch tensors. The tensors are moved to the selected computing device. The model then generates a translated sequence using `model.generate()`. Because the model is being used only for inference, `torch.no_grad()` is used so that PyTorch does not calculate gradients. Finally, the generated token IDs are converted back into readable Romanian text using `tokenizer.decode()`.

The translation function is applied to each English sentence in the selected test sample. The model-generated translations are stored in a predictions list, while the correct Romanian translations from the dataset are stored as references.

For evaluation, the SacreBLEU metric is loaded using `evaluate.load("sacrebleu")`. The `bleu.compute()` command compares the generated translations with the human reference translations and calculates a BLEU score based mainly on the overlap of words and short word sequences between them.

### Evaluation Results and Interpretation

The translation model achieved a BLEU score of **32.75** on the selected sample of 100 English-Romanian test sentences.

This score indicates a moderate level of similarity between the model-generated Romanian translations and the human reference translations. BLEU measures overlap in words and short word sequences between a prediction and its reference, so a higher score generally suggests that the generated translation is closer to the reference.

However, BLEU should not be interpreted as a direct accuracy percentage. A translation can be grammatically correct and preserve the original meaning while still receiving a lower BLEU contribution if it uses different wording from the reference translation.

The sample outputs illustrate this limitation. For example, one sentence was translated exactly the same as the reference, while other sentences expressed similar meanings using different Romanian wording. Some examples also showed awkward phrasing or partial changes in meaning. In addition, the dataset contains subtitle-style or conversational sentences, and some reference translations appear noisy or less literal. This can also influence the BLEU score.

Overall, a BLEU score of 32.75 suggests that the pre-trained English-to-Romanian model performs reasonably well on this test sample, but the translation quality is not perfect. The result also demonstrates why machine translation should be evaluated using both automatic metrics such as BLEU and qualitative inspection of individual translations.

## Conclusion

This lab demonstrated English-to-Romanian machine translation using a pre-trained Transformer-based sequence-to-sequence model from Hugging Face. The workflow included loading the tokenizer and model, preparing an English-Romanian dataset, implementing a reusable `translate()` function, generating translations, and evaluating the results using SacreBLEU.

The model successfully translated the selected English sentences into Romanian without additional training. On a sample of 100 test sentences, the model achieved a BLEU score of **32.75**. The qualitative examples showed that some translations closely matched the references, while others used different wording or contained awkward phrasing.

Overall, the lab demonstrates how pre-trained Transformer models can be applied efficiently to machine translation tasks and how automatic metrics such as BLEU can be combined with manual inspection to better understand translation quality.

## References

Hugging Face. (n.d.). *Transformers documentation*. https://huggingface.co/docs/transformers/

Hugging Face. (n.d.). *Datasets documentation*. https://huggingface.co/docs/datasets/

Hugging Face. (n.d.). *Helsinki-NLP/opus-mt-en-ro*. https://huggingface.co/Helsinki-NLP/opus-mt-en-ro

Helsinki-NLP. (n.d.). *OPUS-100 dataset*. https://huggingface.co/datasets/Helsinki-NLP/opus-100

Post, M. (2018). A call for clarity in reporting BLEU scores. *Proceedings of the Third Conference on Machine Translation: Research Papers*, 186–191.

## AI Use Disclosure

OpenAI ChatGPT was used to support the completion of this lab by helping explain machine translation concepts, developing and reviewing Python code, interpreting the BLEU evaluation results, and improving the clarity of written explanations. All code was executed by the student in the assigned development environment, and the outputs, translations, and evaluation results were reviewed and verified for accuracy before inclusion in the final submission.